# Training Neural Network Skema 2 - Controller Tekanan Pompa

Notebook ini memakai file `DATA_KOMBINASI_21_22.xlsx` untuk latihan skema 2.

**Skema 2** berarti Neural Network diperlakukan sebagai controller. Model menerima kondisi sistem saat ini, lalu memprediksi aksi kontrol berikutnya berupa duty PWM.

Setpoint yang digunakan pada latihan ini:

```text
setpoint = 0.85 bar
```

Catatan penting: kolom `pressure_dummy_bar` pada file Excel adalah data dummy/interpolasi dari data 21 Mei. Ini cocok untuk latihan alur training, tetapi untuk hasil final TA perlu diganti dengan data closed-loop real.

## Penjelasan Skema 2

Pada kontrol tekanan, NN tidak sekadar menebak pressure. NN diberi target `setpoint_bar`, pressure aktual, dan error. Dari informasi itu, NN belajar menentukan duty berikutnya.

Bentuk sederhananya:

```text
Input NN  : setpoint, pressure, error, delta_pressure, previous_duty, voltage, current
Output NN : duty_control
```

Pada versi awal, target output memakai `target_duty_next_percent` dari urutan data. Itu ternyata belum cukup, karena data kombinasi ini bukan data closed-loop asli. Akibatnya NN belajar urutan duty eksperimen, bukan logika kontrol.

Pada versi revisi ini, target output dibuat sebagai `target_control_duty_percent`, yaitu duty hasil aturan kontrol awal:

- Jika pressure jauh di atas setpoint, duty dibuat `0%`.
- Jika pressure sedikit di atas setpoint, duty diturunkan.
- Jika pressure berada dekat setpoint, duty dipertahankan.
- Jika pressure di bawah setpoint, duty dinaikkan.
- Jika motor perlu menyala, duty minimum dibatasi agar tidak berada pada zona motor tidak kuat berputar.

Jadi ini tetap regresi, tetapi targetnya sudah lebih sesuai dengan konsep controller.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True

In [ ]:
DATA_PATH = Path('DATA_KOMBINASI_21_22.xlsx')
SHEET_NAME = 'DATA_KOMBINASI_TRAINING'

df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)
print('Jumlah data:', len(df))
df.head()

In [ ]:
# Kolom penting untuk skema 2.
# setpoint_bar                  : target tekanan, yaitu 0.85 bar
# pressure_dummy_bar            : pressure aktual dummy
# error_bar                     : setpoint - pressure
# delta_pressure_dummy_bar      : perubahan pressure dari sampel sebelumnya
# prev_duty_percent             : duty sebelumnya
# voltage_for_pressure_mapping_v: tegangan JSY yang sudah dikalibrasi
# jsy_current_rms_a             : arus motor

SETPOINT_BAR = 0.85
DEADBAND_BAR = 0.05
SAFETY_SHUTOFF_MARGIN_BAR = 0.08
MIN_RUNNING_DUTY = 55.0
MAX_DUTY = 95.0
KP_DUTY_PER_BAR = 80.0
KD_DUTY_PER_BAR = 35.0

# Pastikan setpoint dan error konsisten.
df['setpoint_bar'] = SETPOINT_BAR
df['error_bar'] = df['setpoint_bar'] - df['pressure_dummy_bar']

# Target kontrol dibuat dari aturan awal.
# Ini bukan pengganti controller final, tapi target latihan NN agar model belajar aksi yang masuk akal.
control_target = []
for _, row in df.iterrows():
    pressure = row['pressure_dummy_bar']
    error = SETPOINT_BAR - pressure
    delta_p = row['delta_pressure_dummy_bar']
    prev_duty = row['prev_duty_percent']

    if pressure >= SETPOINT_BAR + SAFETY_SHUTOFF_MARGIN_BAR:
        duty_cmd = 0.0
    elif abs(error) <= DEADBAND_BAR:
        duty_cmd = prev_duty
    else:
        duty_cmd = prev_duty + KP_DUTY_PER_BAR * error - KD_DUTY_PER_BAR * delta_p

    duty_cmd = float(np.clip(duty_cmd, 0.0, MAX_DUTY))

    # Hindari duty tanggung: kalau motor perlu hidup, jangan beri duty di bawah batas running.
    if 0.0 < duty_cmd < MIN_RUNNING_DUTY and pressure < SETPOINT_BAR - DEADBAND_BAR:
        duty_cmd = MIN_RUNNING_DUTY

    control_target.append(duty_cmd)

df['target_control_duty_percent'] = control_target

input_cols = [
    'setpoint_bar',
    'pressure_dummy_bar',
    'error_bar',
    'delta_pressure_dummy_bar',
    'prev_duty_percent',
    'voltage_for_pressure_mapping_v',
    'jsy_current_rms_a',
]

target_col = 'target_control_duty_percent'

model_df = df[input_cols + [target_col]].dropna().copy()
model_df.head()

In [ ]:
plt.plot(df['elapsed_s'], df['pressure_dummy_bar'], label='Pressure dummy')
plt.axhline(0.85, color='red', linestyle='--', label='Setpoint 0.85 bar')
plt.xlabel('Waktu (s)')
plt.ylabel('Pressure (bar)')
plt.title('Pressure Dummy terhadap Setpoint')
plt.legend()
plt.show()

plt.plot(df['elapsed_s'], df['duty_round_percent'], label='Duty')
plt.plot(df['elapsed_s'], df['voltage_for_pressure_mapping_v'], label='Voltage calibrated')
plt.xlabel('Waktu (s)')
plt.title('Duty dan Tegangan Kalibrasi')
plt.legend()
plt.show()

In [ ]:
X_raw = model_df[input_cols].to_numpy(dtype=np.float32)
y_raw = model_df[[target_col]].to_numpy(dtype=np.float32)

# Split time-series: 80% train, 20% test. Data tidak diacak agar urutan waktu tetap terjaga.
split_idx = int(0.8 * len(model_df))
X_train_raw, X_test_raw = X_raw[:split_idx], X_raw[split_idx:]
y_train_raw, y_test_raw = y_raw[:split_idx], y_raw[split_idx:]

X_min = X_train_raw.min(axis=0)
X_max = X_train_raw.max(axis=0)
X_range = X_max - X_min
X_range[X_range == 0] = 1

y_min = y_train_raw.min(axis=0)
y_max = y_train_raw.max(axis=0)
y_range = y_max - y_min
y_range[y_range == 0] = 1

X_train = (X_train_raw - X_min) / X_range
X_test = (X_test_raw - X_min) / X_range
y_train = (y_train_raw - y_min) / y_range
y_test = (y_test_raw - y_min) / y_range

print('Train:', X_train.shape, 'Test:', X_test.shape)

In [ ]:
# Neural Network manual: 7 input -> 16 hidden -> 8 hidden -> 1 output
rng = np.random.default_rng(85)

n_input = X_train.shape[1]
n_hidden1 = 16
n_hidden2 = 8
n_output = 1

W1 = rng.normal(0, 0.2, size=(n_input, n_hidden1)).astype(np.float32)
b1 = np.zeros((1, n_hidden1), dtype=np.float32)
W2 = rng.normal(0, 0.2, size=(n_hidden1, n_hidden2)).astype(np.float32)
b2 = np.zeros((1, n_hidden2), dtype=np.float32)
W3 = rng.normal(0, 0.2, size=(n_hidden2, n_output)).astype(np.float32)
b3 = np.zeros((1, n_output), dtype=np.float32)

def forward(X):
    z1 = X @ W1 + b1
    a1 = np.tanh(z1)
    z2 = a1 @ W2 + b2
    a2 = np.tanh(z2)
    y = a2 @ W3 + b3
    return z1, a1, z2, a2, y

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

In [ ]:
learning_rate = 0.03
epochs = 5000
loss_history = []
N = X_train.shape[0]

for epoch in range(epochs):
    z1, a1, z2, a2, y_pred = forward(X_train)
    loss = mse(y_train, y_pred)
    loss_history.append(loss)

    dy = (2 / N) * (y_pred - y_train)
    dW3 = a2.T @ dy
    db3 = np.sum(dy, axis=0, keepdims=True)

    da2 = dy @ W3.T
    dz2 = da2 * (1 - np.tanh(z2) ** 2)
    dW2 = a1.T @ dz2
    db2 = np.sum(dz2, axis=0, keepdims=True)

    da1 = dz2 @ W2.T
    dz1 = da1 * (1 - np.tanh(z1) ** 2)
    dW1 = X_train.T @ dz1
    db1 = np.sum(dz1, axis=0, keepdims=True)

    W3 -= learning_rate * dW3
    b3 -= learning_rate * db3
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

    if epoch % 500 == 0:
        print(f'Epoch {epoch:4d} | Loss {loss:.8f}')

print(f'Epoch {epochs:4d} | Loss {loss_history[-1]:.8f}')

In [ ]:
plt.plot(loss_history)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Loss Training NN Controller')
plt.show()

In [ ]:
_, _, _, _, y_pred_test_norm = forward(X_test)
y_pred_test = y_pred_test_norm * y_range + y_min

result = model_df.iloc[split_idx:].copy().reset_index(drop=True)
result['pred_duty_control_percent_raw'] = y_pred_test[:, 0]
result['pred_duty_control_percent'] = np.clip(result['pred_duty_control_percent_raw'], 0, MAX_DUTY)
result['duty_error_percent'] = result['pred_duty_control_percent'] - result[target_col]

mae = np.mean(np.abs(result['duty_error_percent']))
rmse = np.sqrt(np.mean(result['duty_error_percent'] ** 2))
print('MAE duty :', round(mae, 3), '%')
print('RMSE duty:', round(rmse, 3), '%')
result.head(10)

In [ ]:
plt.plot(result.index, result[target_col], label='Target control duty')
plt.plot(result.index, result['pred_duty_control_percent'], label='Prediksi NN', linestyle='--')
plt.xlabel('Index data testing')
plt.ylabel('Duty (%)')
plt.title('Prediksi Duty Kontrol')
plt.legend()
plt.show()

In [ ]:
def predict_duty_next(setpoint_bar, pressure_bar, prev_pressure_bar, prev_duty_percent, voltage_calibrated_v, current_a):
    # Safety override: kalau pressure sudah melewati batas atas, motor dimatikan.
    # Ini sengaja dibuat di luar NN agar perilaku kritis tidak bergantung pada hasil prediksi.
    if pressure_bar >= setpoint_bar + SAFETY_SHUTOFF_MARGIN_BAR:
        return 0.0

    error_bar = setpoint_bar - pressure_bar
    delta_pressure = pressure_bar - prev_pressure_bar
    x = np.array([[setpoint_bar, pressure_bar, error_bar, delta_pressure, prev_duty_percent, voltage_calibrated_v, current_a]], dtype=np.float32)
    x_norm = (x - X_min) / X_range
    _, _, _, _, y_norm = forward(x_norm)
    duty = float((y_norm * y_range + y_min)[0, 0])
    duty = float(np.clip(duty, 0, MAX_DUTY))

    # Hindari duty tanggung yang tidak cukup untuk memutar motor.
    if 0.0 < duty < MIN_RUNNING_DUTY and pressure_bar < setpoint_bar - DEADBAND_BAR:
        duty = MIN_RUNNING_DUTY

    return duty

# Contoh simulasi:
for pressure in [0.50, 0.70, 0.85, 0.95, 1.10]:
    duty_cmd = predict_duty_next(
        setpoint_bar=0.85,
        pressure_bar=pressure,
        prev_pressure_bar=pressure - 0.02,
        prev_duty_percent=60,
        voltage_calibrated_v=124,
        current_a=0.2,
    )
    print(f'Pressure {pressure:.2f} bar -> prediksi duty {duty_cmd:.2f}%')

In [ ]:
def c_array(name, array):
    arr = np.asarray(array, dtype=np.float32)
    if arr.ndim == 1:
        values = ', '.join(f'{v:.8f}f' for v in arr)
        return f'static const float {name}[{arr.shape[0]}] = {{ {values} }};'
    rows = []
    for row in arr:
        values = ', '.join(f'{v:.8f}f' for v in row)
        rows.append('    { ' + values + ' }')
    body = ',
'.join(rows)
    return f'static const float {name}[{arr.shape[0]}][{arr.shape[1]}] = {{
{body}
}};'

print(c_array('X_MIN', X_min))
print(c_array('X_RANGE', X_range))
print(c_array('Y_MIN', y_min.reshape(-1)))
print(c_array('Y_RANGE', y_range.reshape(-1)))
print(c_array('W1', W1))
print(c_array('B1', b1.reshape(-1)))
print(c_array('W2', W2))
print(c_array('B2', b2.reshape(-1)))
print(c_array('W3', W3))
print(c_array('B3', b3.reshape(-1)))

## Catatan Penting untuk TA

Skema ini adalah latihan awal controller NN. Revisi pentingnya adalah target output tidak lagi memakai urutan duty eksperimen mentah, tetapi memakai `target_control_duty_percent`, yaitu target duty yang dibuat dari aturan kontrol awal berbasis error setpoint 0.85 bar.

Alasan revisi:

- Data kombinasi ini belum closed-loop asli.
- Jika target memakai duty berikutnya dari urutan data, NN hanya belajar urutan eksperimen.
- Pada kondisi pressure tinggi seperti 1.10 bar, controller seharusnya mematikan atau menurunkan duty, bukan memberi duty tanggung.

Karena itu fungsi `predict_duty_next()` juga diberi safety override:

```text
jika pressure >= setpoint + 0.08 bar, duty = 0%
```

Untuk data final, lakukan logging closed-loop real dengan kolom minimal:

- `setpoint_bar`
- `pressure_bar`
- `error_bar`
- `delta_pressure_bar`
- `previous_duty_percent`
- `duty_control_percent`
- `jsy_voltage_rms_v` atau tegangan kalibrasi
- `valve_open_count`